# Chapter 14. Antenna Arrays and Beam Steering

The code from Chapter 14 of *Radio Propagation Ray Tracing with Sionna RT*.
The cells follow the order of the text, and the contents of each code cell are
the same as printed there.

Only the first Setup cell is not printed in the book. It performs the imports
and the scene loading that the printed code assumes. Once the setup cell has
run, the rest can be run from the top in order.

Fragments the text shows for explanation, such as a path to a scene of your own
or an object name that scene does not contain, will not run as they stand. A
note appears before each such cell.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# the central Munich urban scene used in many chapters of this book
scene = load_scene(scene_assets.munich)
scene.frequency = 3.5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-30.0, 38.0, 35.0]))

solver = PathSolver()
rmsolver = RadioMapSolver()

## 2. Setting Up a Planar Array

In [ ]:
scene.tx_array = PlanarArray(num_rows=4, num_cols=4,
                              vertical_spacing=0.5,
                              horizontal_spacing=0.5,
                              pattern='tr38901',
                              polarization='V')

## 5. Changing the Beam Direction

In [ ]:
import numpy as np
import mitsuba as mi

# steer to azimuth phi and elevation theta in local coordinates
phi, theta = np.deg2rad(30.0), np.deg2rad(0.0)
direction = np.array([np.cos(theta)*np.cos(phi),
                      np.cos(theta)*np.sin(phi),
                      np.sin(theta)])
pos = np.array(scene.tx_array.normalized_positions)  # (3, N), in wavelengths
phase = 2*np.pi * (pos.T @ direction)
w = np.exp(1j*phase) / np.sqrt(phase.size)
precoding_vec = (mi.TensorXf(w.real), mi.TensorXf(w.imag))
rm = rmsolver(scene, max_depth=3, cell_size=[2.0, 2.0],
              precoding_vec=precoding_vec)